# What Makes an LLM 'Large'?

In this notebook we'll build intuition for the word **"large"** in Large Language Model by:

1. Counting parameters in tiny toy Transformer configurations ourselves
2. Approximating the parameter counts of real models (GPT-2, GPT-3, Llama 2)
3. Visualizing how parameter count scales with depth and width
4. Estimating how much memory a model of a given size needs just to load

No GPU is required — everything here runs on CPU in a few seconds.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print('Torch version:', torch.__version__)
print('Running on CPU — no GPU needed for this lesson.')

## Step 1: Count parameters in a real (tiny) Transformer block

Instead of just quoting a formula, let's build a single Transformer decoder layer with PyTorch and literally count its weights. This is the same building block that GPT-2, GPT-3, and Llama all stack dozens or hundreds of times.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def make_transformer_layer(d_model, n_heads):
    return nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=n_heads,
        dim_feedforward=4 * d_model,
        batch_first=True,
    )

# A tiny layer, similar in spirit to GPT-2 Small's building block
layer = make_transformer_layer(d_model=768, n_heads=12)
print(f'Parameters in ONE Transformer layer (d_model=768): {count_params(layer):,}')
print(f'GPT-2 Small stacks 12 of these -> roughly {count_params(layer) * 12:,} params from layers alone')

## Step 2: Approximate real model sizes with a formula

Manually building a 96-layer, 12,288-wide GPT-3 would take a while to instantiate on CPU. Instead we use the well-known approximation:

```
parameters ≈ 12 × n_layers × d_model²  +  vocab_size × d_model
```

Let's check how close this gets to the real, published parameter counts.

In [ ]:
def transformer_params(n_layers, d_model, vocab_size=50257):
    per_layer = 12 * d_model**2
    embedding = vocab_size * d_model
    return n_layers * per_layer + embedding

models = {
    'GPT-2 Small':  {'n_layers': 12, 'd_model': 768,   'actual': 124_000_000},
    'GPT-2 XL':     {'n_layers': 48, 'd_model': 1600,  'actual': 1_500_000_000},
    'GPT-3':        {'n_layers': 96, 'd_model': 12288, 'actual': 175_000_000_000},
    'Llama 2 7B':   {'n_layers': 32, 'd_model': 4096,  'actual': 7_000_000_000},
    'Llama 2 70B':  {'n_layers': 80, 'd_model': 8192,  'actual': 70_000_000_000},
}

for name, cfg in models.items():
    est = transformer_params(cfg['n_layers'], cfg['d_model'])
    print(f"{name:14s} estimated={est:>15,.0f}   actual={cfg['actual']:>15,.0f}   ratio={est/cfg['actual']:.2f}x")

## Step 3: Visualize the scale jump

A bar chart makes the jump from millions to billions viscerally obvious — note we need a log scale, or GPT-2 Small would be an invisible sliver next to GPT-3.

In [ ]:
names = list(models.keys())
actual_values = [models[n]['actual'] for n in names]
colors = ['#4cc9f0', '#4895ef', '#f72585', '#7209b7', '#b5179e']

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

bars = ax.bar(names, actual_values, color=colors)
ax.set_yscale('log')
ax.set_ylabel('Parameters (log scale)', color='white')
ax.set_title("How 'Large' Grew Over Time", color='white', fontsize=14)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#444444')

for bar, val in zip(bars, actual_values):
    ax.text(bar.get_x() + bar.get_width()/2, val * 1.3, f'{val:,.0f}',
            ha='center', color='white', fontsize=8, rotation=0)

plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Step 4: Experiment — how memory grows with parameters

Every parameter needs to be stored somewhere. In 16-bit precision (2 bytes/parameter), a model's raw weight file size is roughly `parameters × 2 bytes`. Try changing `n_layers` and `d_model` below and see how quickly the required memory explodes — this is exactly why large models need multiple GPUs just to load.

In [ ]:
def memory_gb(n_layers, d_model, bytes_per_param=2, vocab_size=50257):
    params = transformer_params(n_layers, d_model, vocab_size)
    return params, params * bytes_per_param / 1e9

# --- Try changing these two numbers and re-run ---
n_layers = 48
d_model = 1600
# ---------------------------------------------

params, gb = memory_gb(n_layers, d_model)
print(f'Config: {n_layers} layers, d_model={d_model}')
print(f'Estimated parameters: {params:,.0f}')
print(f'Estimated memory to load in fp16: {gb:.2f} GB')
print()
print('A single consumer GPU (e.g. 24 GB) can comfortably hold this model:' , 'YES' if gb < 24 else 'NO — needs multiple GPUs or quantization')

In [ ]:
# Sweep d_model to see the quadratic growth in parameters (and memory)
d_models = np.array([256, 512, 768, 1024, 2048, 4096, 8192, 12288])
fixed_layers = 24
sweep_params = [transformer_params(fixed_layers, d) for d in d_models]
sweep_gb = [p * 2 / 1e9 for p in sweep_params]

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

ax.plot(d_models, sweep_gb, marker='o', color='#4cc9f0', linewidth=2)
ax.set_xlabel('d_model (hidden width)', color='white')
ax.set_ylabel('Memory to load, fp16 (GB)', color='white')
ax.set_title(f'Memory Grows ~Quadratically with Width ({fixed_layers} layers fixed)', color='white')
ax.tick_params(colors='white')
ax.grid(alpha=0.2)
for spine in ax.spines.values():
    spine.set_color('#444444')

plt.tight_layout()
plt.show()

print('Notice: doubling d_model roughly quadruples memory — width is expensive!')

## Recap & Experiments to Try

- We confirmed that "large" is measurable: parameters, training tokens, and compute.
- Our simple formula (`12 × n_layers × d_model² + vocab_size × d_model`) gets remarkably close to real published model sizes.
- Width (`d_model`) is far more expensive than depth (`n_layers`) because it appears squared in the formula.

**Try it yourself:**
1. In Step 4, set `n_layers=96, d_model=12288` to reproduce GPT-3's memory footprint — how many 80GB GPUs would you need?
2. Fix `d_model` and sweep `n_layers` instead — confirm that memory grows linearly, not quadratically, with depth.
3. Look up Chinchilla's "20 tokens per parameter" rule — compute how many training tokens Llama 2 7B would need under that rule, and compare to its publicly reported training set size.